Incorporating the δ term to the instrumental-variable moment expressions

In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import GMM

# Load the data using a simple raw text path
file_path = 'midterm_partone.csv'
input_table = pd.read_csv(file_path)

# First-stage regression (OLS)
model_iv = sm.OLS(input_table["Inventory Turnover"], 
                  input_table[["Constant", "Current Ratio", "Quick Ratio", "Debt Asset Ratio"]]).fit()
endog_predict = model_iv.predict(input_table[["Constant", "Current Ratio", "Quick Ratio", "Debt Asset Ratio"]])
input_table["Endogenous Param"] = endog_predict

# Second-stage regression (OLS)
model_2sls = sm.OLS(input_table["Stock Change"], 
                    input_table[["Constant", "Endogenous Param", "Operating Profit", "Interaction Effect"]]).fit()
print(model_2sls.summary())

# Set up data for GMM
y_vals = np.array(input_table["Stock Change"])
x_vals = np.array(input_table[["Inventory Turnover", "Operating Profit", "Interaction Effect"]])
iv_vals = np.array(input_table[["Current Ratio", "Quick Ratio", "Debt Asset Ratio"]])

# Define a custom GMM model incorporating delta with five parameters
class gmm_with_delta(GMM):
    def __init__(self, *args, delta=0, **kwargs):
        super().__init__(*args, **kwargs)
        self.delta = delta
    
    def momcond(self, params):
        p0, p1, p2, p3, p4 = params  # Adding p4 to the parameter list
        endog = self.endog
        exog = self.exog
        inst = self.instrument

        # Errors from regression equation
        error = endog - p0 - p1 * exog[:, 0] - p2 * exog[:, 1] - p3 * exog[:, 2] - p4 * self.delta

        # Moment conditions with the delta term included
        error0 = error
        error1 = error * exog[:, 1]
        error2 = error * exog[:, 2]
        error3 = error * inst[:, 0]
        error4 = error * inst[:, 1]
        error5 = error * inst[:, 2]

        g = np.column_stack((error0, error1, error2, error3, error4, error5))
        return g

# Initial parameter values for GMM with five parameters
beta0 = np.array([0.1, 0.1, 0.1, 0.1, 0.1]) 

# Fit the GMM model incorporating delta
# Assuming a non-zero value for delta (as per the expert's claim)
delta = 0.05
res = gmm_with_delta(endog=y_vals, exog=x_vals, instrument=iv_vals, k_moms=6, k_params=5, delta=delta).fit(beta0)

# Print the summary of the GMM results
print(res.summary())


                            OLS Regression Results                            
Dep. Variable:           Stock Change   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.013
Method:                 Least Squares   F-statistic:                     8.530
Date:                Sun, 10 Nov 2024   Prob (F-statistic):           1.27e-05
Time:                        12:32:33   Log-Likelihood:                -1186.5
No. Observations:                1696   AIC:                             2381.
Df Residuals:                    1692   BIC:                             2403.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Constant              -0.0176      0